# K03 — ANES synthetic respondents (F03) + detector scores (F04) on Kaggle

One unattended run finishes F03 for every model (resuming from earlier caches) and then runs
F04. Version of 10 November 2026: a model that fails no longer stops the run — its error is
saved to `k03_errors.json` and the detector scores are computed for everything that exists.

**Inputs (one Private dataset, e.g. `d2vi-anes-inputs`):** the two ANES files
(`anes_timeseries_2024_csv_20260519.zip`, `anes_timeseries_2024_redactedopenends_excel_20250923.xlsx`)
and **every file of the previous K03 Output** (`cache_*.csv`, `anes_synth_*.parquet`,
`anes_synth_*.manifest.json`, and `f04_scores_cache.csv` if present).

**Settings:** *Accelerator* = GPU T4 x2, *Internet* = On.

**Run:** *Save Version* → *Save & Run All (Commit)*. The last cell prints `RESULT: COMPLETE`
or `RESULT: PARTIAL` with what is missing. Download the whole Output either way.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes openpyxl
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPUs:", torch.cuda.device_count())
assert d2vi.__version__ >= "2.0.0.dev7"

In [ ]:
import glob, os, shutil, json, hashlib
import pandas as pd
WORK = "/kaggle/working"
inputs = glob.glob("/kaggle/input/**/*", recursive=True)
inputs = [f for f in inputs if os.path.isfile(f)]
RESUME = ("cache_", "f04_scores_cache", "anes_synth_")   # earlier progress, copied once
for f in inputs:
    name = os.path.basename(f)
    if name.startswith(RESUME) and name.endswith((".csv", ".parquet", ".json")):
        dst = os.path.join(WORK, name)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
print("working dir:", sorted(os.listdir(WORK)))
# Kaggle unzips uploaded zip files, so accept the zip or the extracted csv
csv = sorted(f for f in inputs if os.path.basename(f).startswith("anes_timeseries_2024_csv")
             and f.endswith((".zip", ".csv")))
xlsx = sorted(f for f in inputs if os.path.basename(f).startswith("anes_timeseries_2024_redactedopenends")
              and f.endswith(".xlsx"))
assert csv and xlsx, "ANES main data (csv or zip) or open-ends (xlsx) missing from the input dataset"
csv, xlsx = csv[0], xlsx[0]
print("ANES files:", csv, xlsx)
from d2vi.data.anes import ANES2024 as ANESFiles
from d2vi.synthetic import ANES2024_CODEBOOK
anes = ANESFiles(csv, xlsx).human_pool()
assert len(anes) == 4926
ANES2024_CODEBOOK.validate(anes)
print("human pool", anes.shape)


In [ ]:
# F03: synthetic respondents; same seeds as before, so caches resume exactly
import traceback
from d2vi.synthetic import ANES2024, RespondentPrompt, sample_personas
from d2vi.synthetic.generator import HFChatGenerator, SyntheticSurvey, GENERATORS
from d2vi.features.pipeline import package_versions
PER_CONDITION = 400
CONDITIONS = [("demographic", "plain"), ("demographic", "humanlike"),
              ("political", "plain"), ("political", "humanlike")]
ERRORS = {}

def run_model(model_idx, MODEL, out):
    SLUG = MODEL.split("/")[-1]
    gen = HFChatGenerator(MODEL, batch_size=16, max_new_tokens=450)
    try:
        tables = []
        for c, (level, style) in enumerate(CONDITIONS):
            seed = 20261109 + 100 * model_idx + c
            personas = sample_personas(anes, PER_CONDITION, seed=seed)
            prompt = RespondentPrompt(ANES2024, level, style)
            survey = SyntheticSurvey(ANES2024, prompt, f"{WORK}/cache_{SLUG}_{level}_{style}.csv", seed=seed)
            t = survey.run(gen, personas, prefix=f"{SLUG}-{level}-{style}")
            print(SLUG, level, style, "parsed:", round(t.parse_ok.mean(), 3), flush=True)
            tables.append(t)
        synth = pd.concat(tables, ignore_index=True)
        template = RespondentPrompt(ANES2024, "political", "plain").messages(sample_personas(anes, 1, seed=0)[0])
        manifest = {"generator": gen.settings, "conditions": CONDITIONS, "per_condition": PER_CONDITION,
                    "parsed": int(synth.parse_ok.sum()), "rows": len(synth), "instrument": ANES2024.name,
                    "wording_checked": ANES2024.wording_checked,
                    "prompt_sha256": hashlib.sha256(json.dumps(template).encode()).hexdigest(),
                    "anes_files": [os.path.basename(csv), os.path.basename(xlsx)], "persona_pool": len(anes),
                    "platform": "kaggle",
                    "versions": package_versions("d2vi", "transformers", "torch", "bitsandbytes")}
        synth.to_parquet(out, index=False)
        json.dump(manifest, open(out.replace(".parquet", ".manifest.json"), "w"), indent=2)
        print("saved", out, flush=True)
    finally:
        gen.close()

for model_idx, MODEL in enumerate(GENERATORS):
    out = f"{WORK}/anes_synth_{MODEL.split('/')[-1]}.parquet"
    if os.path.exists(out):
        print("done already:", MODEL); continue
    try:
        run_model(model_idx, MODEL, out)
    except Exception as err:
        ERRORS[MODEL] = {"error": repr(err)[:2000], "traceback": traceback.format_exc()[-6000:]}
        print("FAILED:", MODEL, "\n", ERRORS[MODEL]["traceback"], flush=True)
    import gc; gc.collect(); torch.cuda.empty_cache()
json.dump(ERRORS, open(f"{WORK}/k03_errors.json", "w"), indent=2)
print("free GPU memory (GB):", [round(torch.cuda.mem_get_info(i)[0] / 1e9, 1) for i in range(torch.cuda.device_count())])

In [ ]:
# F04: detector scores for every human document and every synthetic document that exists
from d2vi.features.neural import RobertaOpenAIDetector, FastDetectGPT
synth_files = sorted(glob.glob(f"{WORK}/anes_synth_*.parquet"))
COMPLETE = len(synth_files) == len(GENERATORS)
synth = pd.concat([pd.read_parquet(f) for f in synth_files], ignore_index=True)
synth = synth[synth.parse_ok & synth.text.fillna("").str.len().gt(0)]
docs = pd.concat([
    pd.DataFrame({"doc_id": anes.doc_id, "a": 0.0, "source": "human", "text": anes.text}),
    pd.DataFrame({"doc_id": synth.resp_id, "a": 1.0, "source": synth.generator, "text": synth.text}),
], ignore_index=True)
assert docs.doc_id.is_unique
CACHE = f"{WORK}/f04_scores_cache.csv"
done = pd.read_csv(CACHE) if os.path.exists(CACHE) else pd.DataFrame(columns=["doc_id", "score_roberta", "score_fdgpt"])
todo = docs[~docs.doc_id.isin(done.doc_id)]
print("cached", len(done), "| to score", len(todo), flush=True)
if len(todo):
    rob, fd = RobertaOpenAIDetector(batch_size=64), FastDetectGPT()
    for s in range(0, len(todo), 250):
        part = todo.iloc[s:s + 250]
        r = rob.predict(part.text.tolist())
        f = [fd.score_one(t) for t in part.text]
        done = pd.concat([done, pd.DataFrame({"doc_id": part.doc_id, "score_roberta": r, "score_fdgpt": f})], ignore_index=True)
        done.to_csv(CACHE, index=False)
        print(len(done), "scored", flush=True)
scores = docs[["doc_id", "a", "source"]].merge(done, on="doc_id", how="left")
assert scores.score_roberta.notna().all()
print(scores.groupby("source")[["score_roberta", "score_fdgpt"]].mean().round(3))
if COMPLETE:
    scores.to_parquet(f"{WORK}/anes_detector_scores.parquet", index=False)
    json.dump({"rows": len(scores), "by_source": scores.source.value_counts().to_dict(),
               "detectors": {"score_roberta": RobertaOpenAIDetector.model_id,
                             "score_fdgpt": "Fast-DetectGPT analytic, EleutherAI/gpt-neo-2.7B, 400 tokens"},
               "synthetic_files": [os.path.basename(f) for f in synth_files],
               "versions": package_versions("d2vi", "transformers", "torch")},
              open(f"{WORK}/anes_detector_scores.manifest.json", "w"), indent=2)
print(sorted(os.listdir(WORK)))
print("RESULT:", "COMPLETE" if COMPLETE else f"PARTIAL - failed models: {sorted(ERRORS)} (see k03_errors.json); "
      "scores so far are in f04_scores_cache.csv")